
### Part 2: Multi-Source RAG with Routing (100 points)

Build a RAG system that intelligently routes queries to different data sources based on the question type.

**The Data:**

The `data/` folder contains two types of data sources simulating an e-commerce analytics scenario:

```
data/
├── structured/
│   └── daily_sales.csv      # 1000 rows of sales transactions
└── unstructured/
    ├── ELEC001_product_page.txt   # Product descriptions & reviews
    ├── HOME003_product_page.txt
    ├── SPRT001_product_page.txt
    ├── BEAU001_product_page.txt
    ├── CLTH001_product_page.txt
    ├── BOOK001_product_page.txt
    ├── TOYS001_product_page.txt
    ├── OFFC001_product_page.txt
    ├── PETS001_product_page.txt
    └── FOOD001_product_page.txt
```

**Structured Data (CSV):**
- 1000 daily sales records across 90 days (Oct-Dec 2024)
- 35 products across 10 categories
- Columns: `date, product_id, product_name, category, units_sold, unit_price, total_revenue, region`
- Regions: North, South, East, West, Central

**Unstructured Data (Text):**
- 10 product pages with detailed descriptions, specifications, and customer reviews
- Similar to what you'd find on an e-commerce website
- Includes product features, technical specs, and 5 customer reviews per product

**System Architecture:**

```
┌─────────────┐     ┌──────────────────┐     ┌─────────────────┐     ┌─────────────┐
│ User Query  │────>│  Query Router    │────>│  Data Source(s) │────>│ LLM Answer  │
│             │     │ (classify query) │     │  CSV | Text     │     │ with Context│
└─────────────┘     └──────────────────┘     └─────────────────┘     └─────────────┘
                            │
                            ├── Analytical/numerical → CSV (use bash: grep, awk, cut)
                            ├── Product details/reviews → Unstructured text
                            └── Both → Combine sources
```

**Test Questions:**

Your system should be able to answer the following questions:

| # | Question | Source Required |
|---|----------|-----------------|
| 1 | "What was the total revenue for Electronics category in December 2024?" | CSV Only |
| 2 | "Which region had the highest sales volume?" | CSV Only |
| 3 | "What are the key features of the Wireless Bluetooth Headphones?" | Text Only |
| 4 | "What do customers say about the Air Fryer's ease of cleaning?" | Text Only |
| 5 | "Which product has the best customer reviews and how well is it selling?" | Both (Text + CSV) |
| 6 | "I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?" | Both (Text + CSV) |

*Note: Questions 1-2 require filtering/aggregating CSV data. Questions 3-4 require searching unstructured text. Questions 5-6 require combining insights from both sources.*

**What you'll implement:**
- Query router that classifies questions and selects appropriate data source(s)
- CSV retrieval using bash tools (grep, awk, cut) or pandas
- Text retrieval using semantic search or keyword matching
- Result combination strategy for multi-source queries
- Answer generation with the LLM


In [ ]:
import os
import json
import logging
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Setup logging
logging.basicConfig(
    format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s',
    level=logging.INFO
)
logger = logging.getLogger(__name__)
if os.getenv("GROQ_API_KEY"):
    logger.info("GROQ_API_KEY is set")

from langchain_community.chat_models import ChatLiteLLM

# Configure the LLM - using Groq's free Llama model
MODEL_ID = "groq/llama-3.1-8b-instant"

llm = ChatLiteLLM(
    model=MODEL_ID,
    temperature=0
)
logger.info(f"Using model: {MODEL_ID}")

[2026-03-05 23:37:41,807] p4846 {1115267388.py:17} INFO - GROQ_API_KEY is set
[2026-03-05 23:37:41,808] p4846 {1115267388.py:28} INFO - Using model: groq/llama-3.1-8b-instant


In [ ]:
"""
QUERY ROUTING
"""
from sre_parse import State
import time
from langgraph.types import Command

def classify_query(query: str) -> str:
    # Structure questions might use tree or find
    # Code search questions might use grep with file type filters
    # Dependency questions should look at pyproject.toml and package.json
    # Documentation questions should search the docs/ folder

    # use LLM to classify the query  
    system_prompt = """
        You are a query classification tool for a data retrieval system.

        The data is structured as follows:

        Structured Data (CSV), located in `../data/structured/daily_sales.csv`:
        - 1000 daily sales records across 90 days (Oct-Dec 2024)
        - 35 products across 10 categories
        - Columns: `date, product_id, product_name, category, units_sold, unit_price, total_revenue, region`
        - Regions: North, South, East, West, Central

        Unstructured Data (Text), located in `../data/unstructured/`:
        - 10 product pages with detailed descriptions, specifications, and customer reviews
        - Similar to what you'd find on an e-commerce website
        - Includes product features, technical specs, and 5 customer reviews per product

        # Given a user query, determine whether it requires retrieving structured data (CSV), unstructured data (text), or both. If it requires structured data, output a bash command to retrieve the relevant data using `grep` or `awk`. If it requires unstructured data, output a bash command to retrieve the relevant text files using `grep` with appropriate filters. If it requires both, output multiple bash commands separated by a newline.
        
        # Rules:
        - Only return the bash command(s).
        - Do not explain your answer.
        - Do not include any extra text.

        Classify the following query:
    """
        # - Choose exactly ONE category('structured', 'unstructured', 'both').
        # - Output ONLY the category name.

    wait = 1
    for i in range(5):
        time.sleep(wait)
        try:
            response = llm.invoke(f"Prompt: {system_prompt}\nQuestion: {query.lower()}")
            break
        except Exception as e:
            logger.error(f"Error invoking LLM: {e}")
        wait *= 2
    
    return response.content

In [ ]:
"""
BASH TOOL RETRIEVAL
Chunking is not necessary because we do not use vector search. 
Different question types need different retrieval strategies
Structure questions might use tree or find
Code search questions might use grep with file type filters
Dependency questions should look at pyproject.toml and package.json
Documentation questions should search the docs/ folder
"""
import subprocess
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
import re

stop_words = set(stopwords.words("english"))

def extract_keywords(query):
    # words = re.findall(r"\w+", query.lower())
    # return [w for w in words if w not in stop_words and len(w) > 3]
    system_prompt = "You are a retrieval augmented generation AI tool that takes in a query and provides a list of words, phrases, or lemmas that can be searched for in the codebase using 'grep' to find relevant context. Only output the list of words, phrases, or lemmas separated by commas. Do not include any other text. Only include FIVE words, phrases, or lemmas. Use strings that are likely to be found in a codebase. Perform this task for the following query: " 

    wait = 1
    for _ in range(5):
        time.sleep(wait)
        try:
            response = llm.invoke(f"Prompt: {system_prompt}\n{query.lower()}")
            break
        except Exception as e:
            logger.error(f"Error invoking LLM: {e}")
        wait *= 2
    
    query_words = response.content
    query_words = [w.strip() for w in query_words.split(",")]
    return query_words

def run_bash(cmd: str, max_output: int) -> str:
    result = subprocess.run(
        cmd,
        shell=True,
        capture_output=True,
        text=True
    )

    output = result.stdout
    # return output[:max_output]
    return output

# run_bash("cat ../mcp-gateway-registry/pyproject.toml ../mcp-gateway-registry/package.json", max_output=6000)

def retrieve_context(query: str, q_class: str) -> str:
    file_names = []
    bash_cmds = classify_query(query).split("\n")
    context = ""
    for cmd in bash_cmds:
    # if q_class == "structured":
        # cmd = "tree -L 5 ../mcp-gateway-registry"
        # cmds.append(cmd)
        context += run_bash(cmd, max_output=18000//len(bash_cmds))
    # elif q_class == "unstructured":
    
    logger.info(f"\n\nBash command: {cmd}\n")
    logger.info(f"Retrieved context length: {len(context)}\n")
    return [context, bash_cmds, file_names]


[nltk_data] Downloading package stopwords to /home/ubuntu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
import time

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know."
    "\n\n"
)

questions = []
questions.append("What was the total revenue for Electronics category in December 2024?")
questions.append("Which region had the highest sales volume?")
questions.append("What are the key features of the Wireless Bluetooth Headphones?")
questions.append("What do customers say about the Air Fryer's ease of cleaning?")
questions.append("Which product has the best customer reviews and how well is it selling?")
questions.append("I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?")

output_file = "part2_results.txt"
open(output_file, "w").close()

for q in questions:
    wait = 1
    classification = classify_query(q)
    logger.info(f"Classified query as: {classification}")

    context, cmd, file_names = retrieve_context(q, classification)
    # print("\n\n\n")
    print(len(context))
    context = context[:12000]
    # print("\n\n\n")

    for i in range(5):
        time.sleep(wait)
        try:
            response = llm.invoke(
                f"Prompt: {system_prompt}\nContext:\n{context}\n\nQuestion:{q}"
            )
            ans = response.content
            break
        except Exception as e:
            logger.error(f"Error invoking LLM: {e}")

        wait *= 2  

    with open(output_file, "a") as f:
        f.write(f"Question: {q}\n")
        f.write(f"Classification: {classification}\n")  
        f.write(f"Commands: {cmd}\n")
        # f.write(f"Context:\n{contexts[i]}\n")
        # f.write(f"Files: {', '.join(file_names_list[i])}\n")
        f.write(f"Answer: {ans}\n\n")

00:09:22 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 00:09:22,669] p4846 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
00:09:22 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-06 00:09:22,897] p4846 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
[2026-03-06 00:09:22,900] p4846 {2475768031.py:20} INFO - Classified query as: dependencies
[2026-03-06 00:09:22,904] p4846 {1224141644.py:102} INFO - 

Bash command: cat ../mcp-gateway-registry/pyproject.toml ../mcp-gateway-registry/package.json 

[2026-03-06 00:09:22,905] p4846 {1224141644.py:103} INFO - Retrieved context length: 5126



5126


00:09:23 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 00:09:23,907] p4846 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
00:09:24 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-06 00:09:24,473] p4846 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
00:09:25 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 00:09:25,478] p4846 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
00:09:25 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-06 00:09:25,657] p4846 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
[2026-03-06 00:09:25,659] p4846 {2475768031.py:20} INFO - Classified query as: code search
00:09:26 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion(

83707074


00:09:29 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 00:09:29,185] p4846 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
00:09:29 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-06 00:09:29,641] p4846 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
00:09:30 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 00:09:30,645] p4846 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
00:09:30 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-06 00:09:30,881] p4846 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
[2026-03-06 00:09:30,885] p4846 {2475768031.py:20} INFO - Classified query as: file structure
[2026-03-06 00:09:30,888] p4846 {1224141644.py:102} INFO -

...

76130727


00:10:07 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 00:10:07,516] p4846 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 00:10:07,580] p4846 {2475768031.py:37} ERROR - Error invoking LLM: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kex0x720fceb9hdmwxymt5e8` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4804, Requested 2885. Please try again in 16.889999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}




Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers



00:10:09 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 00:10:09,582] p4846 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 00:10:09,646] p4846 {2475768031.py:37} ERROR - Error invoking LLM: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kex0x720fceb9hdmwxymt5e8` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4598, Requested 2885. Please try again in 14.83s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}




Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers

